# Module 4 · Annotation
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 3 · Days 16–20

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M4_Annotation.ipynb)

**By the end of this module:** you can predict variant effects with Ensembl VEP, use population frequencies, check clinical databases and build an annotated table ready to filter.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

### Catch up on earlier modules
This rebuilds the outputs of Modules 1–3 (about 2–4 minutes). Skip it if you ran them in this same session.

In [ ]:
!bash setup/catchup.sh 3

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

## Day 16 · Effect prediction with Ensembl VEP
VEP predicts what each variant does to genes and transcripts (intron, missense, stop-gained, …). We use the public **GRCh37** VEP REST API, because our data are on GRCh37. If the API is busy, the script falls back to a frozen snapshot so the lesson still works.

In [ ]:
!python scripts/benchmark.py 30 10 > /dev/null
!python scripts/annotate.py run/report/alleles_q30_dp10.tsv

In [ ]:
import pandas as pd
ann = pd.read_csv("run/report/annotated.tsv", sep="\t")
display(ann.head(10))
print(ann.consequence.value_counts())

> ✍️ **Your turn**
>
> 1. Which consequence type is most common? Why is that expected in this region?
> 2. Pick one row with an rs ID: give its gene, consequence and why the **build** (GRCh37 vs GRCh38) matters for annotation.

## Day 17 · Population frequency
gnomAD reports how common each allele is across >70,000 genomes. A variant common in healthy people is unlikely to cause a rare disease.

In [ ]:
ann["af"] = pd.to_numeric(ann.gnomad_genomes_af, errors="coerce")
print(ann.frequency_group.value_counts())
ann.af.dropna().plot.hist(bins=20, title="gnomAD genome allele frequency of called variants");

> ✍️ **Your turn**
>
> 1. How many variants are common (≥1%), rare (<1%) or unknown?
> 2. A variant has no gnomAD frequency. Does that prove it is rare? Explain.

## Day 18 · Known variant databases
- **ClinVar**: clinical significance submitted by labs (pathogenic, benign, uncertain).
- **dbSNP**: catalogue of known variants (rs IDs).
- **gnomAD**: population frequencies.

In [ ]:
from IPython.display import Markdown
rows = ann[ann.rs_ids != "."].head(5)
display(Markdown("\n".join(f"- {r.chrom}:{r.pos} {r.ref}>{r.alt} · {r.genes} · [{r.rs_ids}]({r.clinvar_lookup})" for r in rows.itertuples())))

> ✍️ **Your turn**
>
> Open two ClinVar links. For each, record: does the allele (REF/ALT) and build match? Review status? Classification? Any conflicts? If nothing matches, write "no matching assertion found".

## Day 19 · ANNOVAR (optional comparison)
ANNOVAR is another widely used annotation tool. It is free for academic use but requires registration, so it is not installed automatically. After registering at the ANNOVAR website you would run, for example:
```bash
perl table_annovar.pl variants.vcf humandb/ -buildver hg19 -out annotated \
     -protocol refGene,gnomad_genome,clinvar_20240917 -operation g,f,f -vcfinput
```

> ✍️ **Your turn**
>
> List two differences you would expect between VEP and ANNOVAR outputs (hint: transcript sets, consequence terms).

## Day 20 · Build the annotated table

In [ ]:
cols = ["chrom", "pos", "ref", "alt", "gatk", "bcftools", "freebayes", "benchmark", "consequence", "genes", "rs_ids", "gnomad_genomes_af", "frequency_group"]
ann[cols].to_csv("run/report/annotated_clean.tsv", sep="\t", index=False)
print(len(ann), "annotated variants saved to run/report/annotated_clean.tsv")

> ✍️ **Your turn**
>
> **Module 4 deliverable:** a two-row evidence table: allele, gene, consequence, gnomAD frequency, ClinVar evidence and one uncertainty for each.